# Unsloth QLoRA Fine-tuning: Qwen2.5-3B on Free Colab T4

**Target:** Fine-tune Qwen2.5-3B-Instruct on your ACEsi conversation data using 4-bit QLoRA.
**Hardware:** Google Colab free tier (T4 16GB VRAM) — ~30-60 min training.
**Output:** LoRA adapters + merged model → push to Hugging Face Hub → serve via Together/Replicate/HF Inference.

---

In [ ]:
# @title 1️⃣ Install Unsloth & dependencies (run once)
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    "trl<0.9.0" "peft" "accelerate" "bitsandbytes" "datasets" "huggingface_hub" 2>&1 | tail -20

In [ ]:
# @title 2️⃣ Login to Hugging Face (needed to push model)
from huggingface_hub import login

# Paste your HF token (Settings → Access Tokens → New token → write access)
HF_TOKEN = ""  # ← PUT YOUR TOKEN HERE
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=True)
    print("✅ Logged in to Hugging Face")
else:
    print("⚠️ No token provided — you can still train, but cannot push to Hub")

In [ ]:
# @title 3️⃣ Load Qwen2.5-3B-Instruct in 4-bit (QLoRA)
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048  # Context length (reduce if OOM)
dtype = None  # Auto-detect
load_in_4bit = True  # 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",  # Pre-quantized Unsloth build
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank (8-32 typical)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for long context
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
print(f"✅ Model loaded: {model.num_parameters():,} params (trainable: {model.num_parameters(only_trainable=True):,})")

In [ ]:
# @title 4️⃣ Prepare your ACEsi training data
import json
from datasets import Dataset

# ── Option A: Paste your data here as a list of {"user": ..., "assistant": ...} ──
training_data = [
    # Example format — REPLACE WITH YOUR REAL DATA:
    {"user": "what is the oldest book in the Harvard library?",
     "assistant": "The oldest book in the Harvard Library is a copy of the Gutenberg Bible — a 15th-century printed book held in Widener Library.\n\nSource: Harvard Library's own collection page..."},
    {"user": "how do I navigate to the ITS portal in the StudentSyncSA app?",
     "assistant": "Use ui_app_open with package 'com.univen.studentsyncsa', then ui_dump to read the screen..."},
    # Add 500-2000 more examples...
]

# ── Option B: Load from a JSONL file you upload to Colab ──
# training_data = []
# with open("/content/my_acesi_data.jsonl", "r") as f:
#     for line in f:
#         training_data.append(json.loads(line))

def format_chat(example):
    """Convert to chat template format."""
    messages = [
        {"role": "user", "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(training_data).map(format_chat, remove_columns=["user", "assistant"])
print(f"✅ Dataset: {len(dataset)} examples")
print(dataset[0]["text"][:500] + "...")

In [ ]:
# @title 5️⃣ Configure Trainer (SFTTrainer from TRL)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # True for speed if sequences are short
    args=TrainingArguments(
        per_device_train_batch_size=2,  # T4 16GB fits 2x 2048
        gradient_accumulation_steps=4,  # Effective batch = 8
        warmup_steps=10,
        max_steps=100,  # ← CHANGE: 100 steps ≈ 500-1000 examples
        # num_train_epochs=3,  # Alternative to max_steps
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="/content/outputs",
        save_strategy="steps",
        save_steps=50,
        report_to="none",  # or "wandb" if you have W&B
    ),
)
print("✅ Trainer ready")

In [ ]:
# @title 6️⃣ Train! (takes 20-60 min on T4)
trainer_stats = trainer.train()
print(f"✅ Training done in {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
# @title 7️⃣ Save LoRA adapters locally
model.save_pretrained("/content/qwen2.5-3b-lora")
tokenizer.save_pretrained("/content/qwen2.5-3b-lora")
print("✅ LoRA adapters saved to /content/qwen2.5-3b-lora")

In [ ]:
# @title 8️⃣ (Optional) Merge LoRA → full 16-bit model for easy serving
# This creates a standalone model (no PEFT needed at inference)
merged_model = FastLanguageModel.merge_model(model, tokenizer, save_method="merged_16bit")
merged_model.save_pretrained("/content/qwen2.5-3b-merged")
tokenizer.save_pretrained("/content/qwen2.5-3b-merged")
print("✅ Merged model saved to /content/qwen2.5-3b-merged")

In [ ]:
# @title 9️⃣ Push to Hugging Face Hub (requires login from step 2)
HF_REPO = "your-username/qwen2.5-3b-acesi"  # ← CHANGE THIS

if HF_TOKEN:
    # Push LoRA adapters (small, ~50MB)
    model.push_to_hub(HF_REPO + "-lora", token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO + "-lora", token=HF_TOKEN)
    print(f"✅ LoRA pushed to https://huggingface.co/{HF_REPO}-lora")
    
    # Push merged model (larger, ~6GB)
    merged_model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"✅ Merged model pushed to https://huggingface.co/{HF_REPO}")
else:
    print("⚠️ Skipped push — no HF token")

In [ ]:
# @title 🔟 Quick inference test (using merged model)
from unsloth import FastLanguageModel

# Reload merged model for inference
inf_model, inf_tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/qwen2.5-3b-merged",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(inf_model)

test_prompt = "what is the oldest book in the Harvard library?"
messages = [{"role": "user", "content": test_prompt}]
inputs = inf_tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = inf_model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)
response = inf_tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(f"User: {test_prompt}")
print(f"Model: {response}")

---
## 📋 Next Steps After Training

1. **Host the model** (pick one):
   - **Together AI** → `together models upload` the merged model → get OpenAI-compatible endpoint
   - **Hugging Face Inference Endpoints** → one-click deploy from the Hub repo
   - **Replicate** → `replicate push` with a `cog.yaml`
   - **Fireworks AI** → upload via dashboard

2. **Wire into ACEsi**:
   - Add the hosted endpoint as a new provider in `server.py` (like Groq/OpenRouter)
   - Or call directly from ACEsi frontend via `fetch` to your endpoint

3. **Iterate**:
   - Collect more ACEsi conversations → retrain with more steps/epochs
   - Try `r=32`, `max_steps=300`, `learning_rate=1e-4` for better quality

---